[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/probability_distributions.ipynb)

# Probability distributions full course: the practice set, in Python

Thirty-six exercises in four tiers, the same as the practice spreadsheet: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (several steps), **tier 3** stretch problems (combine ideas, choose the distribution, spot the
trap, work backwards) and **tier 4**, the case: all 250 sold-out nights of Halverton Hotels, a 200-room city hotel (a
fictional company; our own data), and the question of how many bookings it should accept. Each tier is harder than
the last.

Under every tier there's a short **how you'd do this in Python** cell: the same calculations with Python's `math`
module, `numpy`, `pandas` and the distributions in `scipy.stats`, so you can use them on your own problems. And a few
cells you can move yourself: the binomial's bars as p changes, the normal's shaded area as b moves, a portfolio's
risk as the correlation turns, and the hotel's gain table as the cost of walking a guest rises.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Probabilities
as decimals: `0.2048`, not `20.48%`. Numbers without thousands separators: `1000`, not `1,000`. Dollars without the
sign: `158`. You can type the arithmetic too: `1 - 38/200`, `exp(-2)`, `sqrt(35)`. For several answers, separate them
with commas, in the order the question lists them (the box's label shows the order). You can also put your answer
between the quotes in a cell's last line, `practice("P1.5", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula and spreadsheet function from the course on two pages:
[probability distributions cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/probability_distributions_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import ast, datetime, math, operator, random, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import binom, poisson, hypergeom, norm, expon, uniform
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # data in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# centre SKY (the mean) · spread VIOLET · flagged CORAL (a walked guest) · the one key mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="our own data · computed"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "ln": math.log, "log": math.log}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets, sqrt(…), exp(…) and ln(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id == "e": return math.e
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"\b(hours?|minutes?|mins?|units?|bookings?|nights?|g|ways)\b|[$%]"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    t = re.sub(UNITS, "", s, flags=re.I).strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-6)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the same questions and answers as the spreadsheet) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Probability essentials',
     "q": "In how many ways can an auditor choose 3 of a company's 10 suppliers to visit, if the order doesn't matter?",
     "parts": ['number of ways'],
     "answer": [120], "tol": 0, "function": 'COMBIN',
     "working": '10 choose 3 = 10 × 9 × 8 / (3 × 2 × 1) = 120'},
    {"id": 'P1.2', "tier": 1, "group": 'Probability essentials',
     "q": 'Of 200 support calls, 38 were about billing. Estimate the probability that a call is NOT about billing.',
     "parts": ['P(not billing)'],
     "answer": [0.81], "tol": 0.001, "function": '—',
     "working": 'P(billing) = 38/200 = 0.19; complement 1 − 0.19 = 0.81'},
    {"id": 'P1.3', "tier": 1, "group": 'Probability essentials',
     "q": 'Of 350 grade-B loans, 21 defaulted. What is P(default | grade B)?',
     "parts": ['P(default | grade B)'],
     "answer": [0.06], "tol": 0.001, "function": '—',
     "working": '21/350 = 0.06'},
    {"id": 'P1.4', "tier": 1, "group": 'Random variables, expected value and variance',
     "q": 'A kiosk sells 0, 1 or 2 extended warranties a day with probabilities 0.5, 0.3 and 0.2. What is the expected number a day?',
     "parts": ['E(X), warranties a day'],
     "answer": [0.7], "tol": 0.001, "function": 'SUMPRODUCT',
     "working": '0 × 0.5 + 1 × 0.3 + 2 × 0.2 = 0.7'},
    {"id": 'P1.5', "tier": 1, "group": 'The binomial distribution',
     "q": 'Each of 5 leads converts with probability 0.2, independently. What is P(exactly 2 convert)?',
     "parts": ['P(X = 2)'],
     "answer": [0.2048000000000001], "tol": 0.0005, "function": 'BINOM.DIST(…, FALSE)',
     "working": '5 choose 2 × 0.2² × 0.8³ = 10 × 0.04 × 0.512 = 0.2048'},
    {"id": 'P1.6', "tier": 1, "group": 'Poisson and hypergeometric',
     "q": 'A plant averages 2 safety incidents a month. With a Poisson model, what is P(no incident next month)?',
     "parts": ['P(no incident)'],
     "answer": [0.1353352832366127], "tol": 0.0005, "function": 'POISSON.DIST(…, FALSE)',
     "working": 'e^−2 = 0.1353'},
    {"id": 'P1.7', "tier": 1, "group": 'Uniform and normal',
     "q": "A report's run time is uniform between 10 and 30 minutes. What is P(it finishes in under 15 minutes)?",
     "parts": ['P(under 15 minutes)'],
     "answer": [0.25], "tol": 0.001, "function": '—',
     "working": '(15 − 10)/(30 − 10) = 0.25'},
    {"id": 'P1.8', "tier": 1, "group": 'Uniform and normal',
     "q": 'Find P(Z ≤ 1.5) for the standard normal.',
     "parts": ['P(Z ≤ 1.5)'],
     "answer": [0.9331927987311419], "tol": 0.0005, "function": 'NORM.S.DIST(z, TRUE)',
     "working": 'table row 1.5, column .00: 0.9332'},
    {"id": 'P1.9', "tier": 1, "group": 'Uniform and normal',
     "q": 'Daily orders are normal with mean 80 and standard deviation 8. What is the z-value of a day with 92 orders?',
     "parts": ['z'],
     "answer": [1.5], "tol": 0.001, "function": 'STANDARDIZE',
     "working": '(92 − 80)/8 = 1.5'},
    {"id": 'P1.10', "tier": 1, "group": 'The exponential distribution',
     "q": 'The time between orders is exponential with mean 10 minutes. What is P(the next order arrives within 5 minutes)?',
     "parts": ['P(within 5 minutes)'],
     "answer": [0.3934693402873666], "tol": 0.0005, "function": 'EXPON.DIST(x, 1/mean, TRUE)',
     "working": '1 − e^(−5/10) = 1 − e^−0.5 = 0.3935'},
    {"id": 'P2.1', "tier": 2, "group": 'Probability essentials',
     "q": '45% of customers use the app, 30% subscribe to the newsletter, 12% do both. What share does at least one?',
     "parts": ['P(at least one)'],
     "answer": [0.63], "tol": 0.001, "function": '—',
     "working": 'P(A ∪ B) = 0.45 + 0.30 − 0.12 = 0.63'},
    {"id": 'P2.2', "tier": 2, "group": 'Probability essentials',
     "q": 'Supplier A ships 70% of a part with 1% defective; supplier B 30% with 4% defective. A part is defective: P(it came from B)?',
     "parts": ['P(B | defective)'],
     "answer": [0.631578947368421], "tol": 0.0005, "function": '—',
     "working": '0.3 × 0.04 = 0.012; 0.7 × 0.01 = 0.007; 0.012/(0.007 + 0.012) = 0.6316'},
    {"id": 'P2.3', "tier": 2, "group": 'Random variables, expected value and variance',
     "q": 'Daily deliveries X take values 1, 2, 3, 4 with probabilities 0.1, 0.4, 0.3, 0.2. Find E(X), Var(X) and σ.',
     "parts": ['E(X)', 'Var(X)', 'σ'],
     "answer": [2.6, 0.84, 0.916515138991168], "tol": 0.001, "function": 'SUMPRODUCT, SQRT',
     "working": 'E = 0.1 + 0.8 + 0.9 + 0.8 = 2.6; Var = Σ(x − 2.6)² f = 0.84; σ = √0.84 = 0.9165'},
    {"id": 'P2.4', "tier": 2, "group": 'Covariance and portfolios',
     "q": "Half in fund X (E 8%, σ 12%) and half in fund Y (E 3%, σ 4%); σxy = −10. Find the portfolio's expected return and σ.",
     "parts": ['expected return (%)', 'σ (%)'],
     "answer": [5.5, 5.916079783099616], "tol": 0.001, "function": 'SQRT',
     "working": 'E = 4 + 1.5 = 5.5 %; Var = 0.25 × 144 + 0.25 × 16 + 2 × 0.25 × (−10) = 35; σ = √35 = 5.916 %'},
    {"id": 'P2.5', "tier": 2, "group": 'The binomial distribution',
     "q": "A machine's parts are 10% defective, independently. In a box of 20, find P(at least 2 defective).",
     "parts": ['P(at least 2 defective)'],
     "answer": [0.6082530018748321], "tol": 0.0005, "function": 'BINOM.DIST(1, 20, 0.1, TRUE)',
     "working": '1 − P(X ≤ 1) = 1 − (0.1216 + 0.2702) = 0.6083'},
    {"id": 'P2.6', "tier": 2, "group": 'The binomial distribution',
     "q": '400 invoices, each with a 10% chance of an error. Find the mean and standard deviation of the number with errors.',
     "parts": ['mean μ', 'σ'],
     "answer": [40, 6], "tol": 0.001, "function": 'SQRT',
     "working": 'μ = np = 40; σ = √(np(1 − p)) = √36 = 6'},
    {"id": 'P2.7', "tier": 2, "group": 'Poisson and hypergeometric',
     "q": 'Calls arrive at 6 an hour (Poisson). Find P(at most 1 call in a 10-minute window).',
     "parts": ['P(at most 1 call)'],
     "answer": [0.7357588823428847], "tol": 0.0005, "function": 'POISSON.DIST(1, 1, TRUE)',
     "working": 'μ = 6 × 10/60 = 1; P(0) + P(1) = e^−1 + e^−1 = 0.7358'},
    {"id": 'P2.8', "tier": 2, "group": 'Poisson and hypergeometric',
     "q": 'A batch of 15 laptops has 4 with a faulty battery. IT takes 3 at random: P(none is faulty)?',
     "parts": ['P(none faulty)'],
     "answer": [0.3626373626373626], "tol": 0.0005, "function": 'HYPGEOM.DIST(0, 3, 4, 15, FALSE)',
     "working": '11 choose 3 / 15 choose 3 = 165/455 = 0.3626'},
    {"id": 'P2.9', "tier": 2, "group": 'Uniform and normal',
     "q": 'Delivery times are normal, mean 50 h, σ 5 h. Find P(45 ≤ X ≤ 60).',
     "parts": ['P(45 ≤ X ≤ 60)'],
     "answer": [0.8185946141203637], "tol": 0.0005, "function": 'NORM.DIST(…, TRUE)',
     "working": 'z = −1 and 2; 0.9772 − 0.1587 = 0.8185 (table), 0.8186 exactly'},
    {"id": 'P2.10', "tier": 2, "group": 'Uniform and normal',
     "q": 'Weekly demand is normal, mean 200 units, σ 20. What demand is exceeded only 10% of weeks?',
     "parts": ['demand exceeded 10% of weeks (units)'],
     "answer": [225.631031310892], "tol": 0.05, "function": 'NORM.INV(0.9, 200, 20)',
     "working": 'z = 1.2816; 200 + 1.2816 × 20 = 225.63'},
    {"id": 'P3.1', "tier": 3, "group": 'Random variables, expected value and variance',
     "q": 'Daily laptop sales X: 0–4 with probabilities 0.10, 0.20, 0.30, 0.25, 0.15. Each sale earns $120; the display costs $100 a day. Find the expected daily profit and its σ.',
     "parts": ['expected daily profit ($)', 'σ of the profit ($)'],
     "answer": [158, 143.37363774418225], "tol": 0.01, "function": 'SUMPRODUCT, SQRT',
     "working": 'E(X) = 2.15, Var(X) = 1.4275; E(120X − 100) = 258 − 100 = $158; σ = 120 × 1.1948 = $143.37'},
    {"id": 'P3.2', "tier": 3, "group": 'Poisson and hypergeometric',
     "q": '5 of a batch of 12 parts are tested; 3 of the 12 are defective. Choose the right distribution and find P(no defective in the 5). What would the binomial wrongly give?',
     "parts": ['P(no defective): the right model', 'what the binomial wrongly gives'],
     "answer": [0.1590909090909091, 0.2373046875], "tol": 0.0005, "function": 'HYPGEOM.DIST(0, 5, 3, 12, FALSE)',
     "working": 'without replacement from a small batch → hypergeometric: 9 choose 5 / 12 choose 5 = 126/792 = 0.1591; the binomial 0.75⁵ = 0.2373 overstates it'},
    {"id": 'P3.3', "tier": 3, "group": 'Uniform and normal',
     "q": 'Bags are filled with mean 500 g; 2.5% weigh more than 539.2 g. If the fill is normal, what is σ?',
     "parts": ['σ (g)'],
     "answer": [20], "tol": 0.01, "function": 'NORM.S.INV(0.975)',
     "working": 'z for the top 2.5 % = 1.96; σ = (539.2 − 500)/1.96 = 20 g'},
    {"id": 'P3.4', "tier": 3, "group": 'Uniform and normal',
     "q": '100 customers each buy with p = 0.3. Approximate P(X ≤ 25) with the normal, with and without the continuity correction; compare with the exact binomial.',
     "parts": ['normal, with the correction', 'normal, without it', 'exact binomial'],
     "answer": [0.1630547260102445, 0.13761676203741713, 0.16313010446635082], "tol": 0.0005, "function": 'NORM.DIST, BINOM.DIST(25, 100, 0.3, TRUE)',
     "working": 'μ = 30, σ = √21 = 4.583; z = (25.5 − 30)/4.583 = −0.98 → 0.1631; without: z = −1.09 → 0.1376; exact 0.1631'},
    {"id": 'P3.5', "tier": 3, "group": 'The exponential distribution',
     "q": 'Orders arrive at 4 an hour. Find P(no order in the next 30 minutes) two ways: Poisson counts and exponential gaps.',
     "parts": ['Poisson: P(0 orders)', 'exponential: P(gap > 30 min)'],
     "answer": [0.1353352832366127, 0.1353352832366127], "tol": 0.0005, "function": 'POISSON.DIST, EXPON.DIST',
     "working": 'Poisson μ = 2: e^−2 = 0.1353; exponential mean 15 min: e^(−30/15) = 0.1353'},
    {"id": 'P3.6', "tier": 3, "group": 'Covariance and portfolios',
     "q": 'A stock fund (σ 14%) and a bond fund (σ² 8.76) in a 50/50 mix: σ was 6.40% with correlation −0.49. What is σ if the correlation were +0.5?',
     "parts": ['σ of the mix (%)'],
     "answer": [7.845320516770753], "tol": 0.005, "function": 'SQRT',
     "working": 'σxy = 0.5 × 14 × 2.96 = 20.72; Var = 49 + 2.19 + 10.36 = 61.55; σ = 7.85 % (diversification shrinks as the correlation rises)'},
    {"id": 'P3.7', "tier": 3, "group": 'Random variables, expected value and variance',
     "q": 'A $2,000 repair happens in a year with probability 0.08. A warranty costs $200. Compare expected costs; at what probability would they be equal?',
     "parts": ['expected repair cost ($)', "probability where they're equal"],
     "answer": [160, 0.1], "tol": 0.001, "function": '—',
     "working": 'expected repair cost 0.08 × 2000 = $160 < $200; equal when p × 2000 = 200, p = 0.10'},
    {"id": 'P3.8', "tier": 3, "group": 'Probability essentials',
     "q": 'A fraud alert catches 98% of fraud and flags 2% of genuine payments. If 2% of payments are fraud, what is P(fraud | alert)?',
     "parts": ['P(fraud | alert)'],
     "answer": [0.5], "tol": 0.001, "function": '—',
     "working": '0.02 × 0.98 = 0.0196; 0.98 × 0.02 = 0.0196; 0.0196/0.0392 = 0.5'},
    {"id": 'P4.1', "tier": 4, "group": 'Case study: Halverton Hotels',
     "q": 'From the 250 sold-out nights, find the mean and the sample variance of no-shows, and the no-show probability p.',
     "parts": ['mean no-shows', 'sample variance', 'p (no-show probability)'],
     "answer": [10, 9.502008032128513, 0.05], "tol": 0.001, "function": 'AVERAGE, VAR.S',
     "working": 'AVERAGE = 10; VAR.S = 9.502; p = 10/200 = 0.05'},
    {"id": 'P4.2', "tier": 4, "group": 'Case study: Halverton Hotels',
     "q": 'If no-shows are binomial(200, 0.05), what variance should the data show? Does the data agree?',
     "parts": ['binomial variance np(1 − p)'],
     "answer": [9.5], "tol": 0.001, "function": 'VAR.S',
     "working": 'np(1 − p) = 200 × 0.05 × 0.95 = 9.5, against 9.502 in the data: guests behave independently enough'},
    {"id": 'P4.3', "tier": 4, "group": 'Case study: Halverton Hotels',
     "q": 'With 208 bookings, and with 210, what is P(more guests arrive than the 200 rooms)?',
     "parts": ['P(walk) with 208 bookings', 'P(walk) with 210 bookings'],
     "answer": [0.17943087615492231, 0.3925830001339069], "tol": 0.0005, "function": 'BINOM.DIST',
     "working": 'P(no-shows ≤ 7) = BINOM.DIST(7, 208, 0.05, TRUE) = 0.1794; P(no-shows ≤ 9) with 210 = 0.3926'},
    {"id": 'P4.4', "tier": 4, "group": 'Case study: Halverton Hotels',
     "q": 'Build the expected-gain table for 0 to 15 extra bookings ($160 a filled room, $450 a walked guest). Which number is best, and what does it earn a night?',
     "parts": ['best number of extra bookings', 'its expected gain ($ a night)'],
     "answer": [8, 997.3684846878684], "tol": 0.5, "function": 'BINOM.DIST, SUMPRODUCT',
     "working": 'gain(k) = 160 × (E[filled] − 190) − 450 × E[walked]; best k = 8: $997 a night'},
    {"id": 'P4.5', "tier": 4, "group": 'Case study: Halverton Hotels',
     "q": 'Sensitivity: if walking a guest really costs $1,000 (lost loyalty included), what is the best number of extra bookings?',
     "parts": ['best number of extra bookings'],
     "answer": [7], "tol": 0, "function": 'BINOM.DIST, SUMPRODUCT',
     "working": 'recompute the table with 1000: the best k falls to 7'},
    {"id": 'P4.6', "tier": 4, "group": 'Case study: Halverton Hotels',
     "q": 'Split the nights: Friday and Saturday nights vs the rest. Compare their mean no-shows and standard deviations. Does one policy fit both?',
     "parts": ['other nights: mean', 'Fri and Sat: mean', 'other nights: s', 'Fri and Sat: s'],
     "answer": [9.857923497267759, 10.388059701492537, 3.1521020466978054, 2.8707259343798603], "tol": 0.01, "function": 'AVERAGEIF, STDEV.S',
     "working": 'AVERAGEIF / STDEV.S by group: the two means and spreads are close (see the answer): one policy fits both'},
    {"id": 'P4.7', "tier": 4, "group": 'Case study: Halverton Hotels',
     "q": 'Walk-in requests per night: find their mean. With a Poisson model at that mean, what is P(at least 2 walk-ins), the chance that two empty rooms could be sold on the spot?',
     "parts": ['mean walk-ins a night', 'P(at least 2 walk-ins)'],
     "answer": [2.376, 0.6862969443842406], "tol": 0.001, "function": 'AVERAGE, POISSON.DIST',
     "working": 'AVERAGE(walk_ins); 1 − POISSON.DIST(1, mean, TRUE)'},
    {"id": 'P4.8', "tier": 4, "group": 'Case study: Halverton Hotels',
     "q": 'The recommendation in numbers a manager can act on: how many extra bookings to accept, the chance a guest is walked on a sold-out night, and the guests walked a night on average. Then say it in one sentence.',
     "parts": ['extra bookings', 'P(a guest is walked), two decimals', 'guests walked a night, two decimals'],
     "answer": [8, 0.18, 0.36], "tol": 0.01, "function": '—',
     "working": 'Accept 208 bookings on sold-out nights: a guest is walked on about 18% of them (0.36 guests a night on average), for about $997 a night more than not overbooking.'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch", 4: "Case"}
assert len(EX) == 36 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8, 8]

def practice(k, answer=""):
    """Exercise k ("P1.5"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, q["tol"]) for g, w in zip(got, q["answer"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""): print(textwrap.fill(f"In a spreadsheet: {q['function']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers practice.py gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def close(a, b, tol=1e-9):
    return all(abs(x - y) <= tol for x, y in zip(a, b)) and len(a) == len(b)

## Tier 1 · Warm-up

One idea, one step, small numbers: a count of combinations, a complement, a conditional probability, an expected
value, one binomial, one Poisson and one uniform probability, the standard normal, a z-value and an exponential gap.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

`math.comb` and `math.perm` count. Every distribution in `scipy.stats` has the same methods: `pmf` (a discrete
distribution's P(X = x), a spreadsheet's FALSE), `pdf` (a continuous one's density), `cdf` (P(X ≤ x), TRUE), `sf`
(P(X > x) = 1 − cdf) and `ppf` (the inverse: NORM.INV). One difference trips people up: `scipy.stats.expon` and
`uniform` take `scale`, which for the exponential is the **mean**, while a spreadsheet's EXPON.DIST takes the rate,
1/mean.

In [ ]:
print("10 choose 3:", math.comb(10, 3), "   ordered:", math.perm(10, 3))
print("complement:", 1 - 38 / 200, "   P(default | B):", 21 / 350)
x, f = np.array([0, 1, 2]), np.array([0.5, 0.3, 0.2])
print("E(X):", (x * f).sum())
print("binomial P(X = 2), n = 5, p = 0.2:", round(binom.pmf(2, 5, 0.2), 4))
print("Poisson P(X = 0), μ = 2:", round(poisson.pmf(0, 2), 4))
print("uniform 10–30, P(X < 15):", uniform.cdf(15, loc=10, scale=20))
print("P(Z ≤ 1.5):", round(norm.cdf(1.5), 4), "   z of 92:", (92 - 80) / 8)
print("exponential mean 10, P(X ≤ 5):", round(expon.cdf(5, scale=10), 4))

assert [math.comb(10, 3)] == answers("P1.1") and close([1 - 38 / 200], answers("P1.2"))
assert close([21 / 350], answers("P1.3")) and close([(x * f).sum()], answers("P1.4"))
assert close([binom.pmf(2, 5, 0.2)], answers("P1.5")) and close([poisson.pmf(0, 2)], answers("P1.6"))
assert close([uniform.cdf(15, loc=10, scale=20)], answers("P1.7")) and close([norm.cdf(1.5)], answers("P1.8"))
assert close([(92 - 80) / 8], answers("P1.9")) and close([expon.cdf(5, scale=10)], answers("P1.10"))

### Change p: the whole distribution slides

The hotel's no-shows: 200 bookings, each guest a no-show with probability p. At p = 0.05 the bars centre on 10 (sky).
Drag p to 0.08 and watch the bars slide right, to a mean of 16, and spread out. The gold bars are x or fewer, the
cumulative P(X ≤ x) that BINOM.DIST gives with TRUE.

In [ ]:
def binomial_bars(n=200, p=0.05, x=9):
    k = np.arange(0, 41)
    f = binom.pmf(k, n, p)
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.15)
    ax.bar(k, f, color=[GOLD if v <= x else CHALK for v in k], width=0.8)
    ax.axvline(n * p, color=SKY, lw=2)
    ax.set_xlim(-0.8, 40.8); ax.set_xlabel("x (no-shows)"); ax.set_ylabel("P(X = x)")
    fig.suptitle(f"B({n}, {p:.3f}):  mean {n * p:.1f},  σ {math.sqrt(n * p * (1 - p)):.2f},  "
                 f"P(X ≤ {x}) = {binom.cdf(x, n, p):.4f}", x=0.07, ha="left", fontsize=15)
    footnote(fig); plt.show()

assert round(binom.cdf(9, 200, 0.05), 4) == 0.4547 and 200 * 0.08 == 16                    # the video's numbers

In [ ]:
interact(binomial_bars, n=IntSlider(value=200, min=10, max=400, step=10, description="n"),
         p=FloatSlider(value=0.05, min=0.01, max=0.15, step=0.005, readout_format=".3f", description="p"),
         x=IntSlider(value=9, min=0, max=40, description="x"));

## Tier 2 · Core

The module's standard problems, several steps each: the addition law, Bayes' theorem, a distribution's mean, variance
and standard deviation, a two-fund portfolio, binomial complements, the binomial's mean and σ, a Poisson rescaled to
a new interval, a hypergeometric draw, a normal probability between two values and a percentile.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Expected value and variance** of a discrete distribution are sums over arrays: `(x * f).sum()` and
`((x - mu) ** 2 * f).sum()`. `scipy.stats.rv_discrete` builds a distribution from your own table and gives `mean()`,
`var()` and `std()`.

**"At least" and "more than"** use the complement: P(X ≥ 2) = 1 − P(X ≤ 1) = `binom.sf(1, n, p)`.

**The hypergeometric's arguments** come in a different order from the spreadsheet's. HYPGEOM.DIST(x, sample, successes
in the population, population, FALSE) is `hypergeom.pmf(x, M=population, n=successes in the population, N=sample)`.

In [ ]:
x, f = np.array([1, 2, 3, 4]), np.array([0.1, 0.4, 0.3, 0.2])                              # P2.3
mu = (x * f).sum(); var = ((x - mu) ** 2 * f).sum()
deliveries = stats.rv_discrete(values=(x, f))
print("E(X):", round(mu, 4), "  Var(X):", round(var, 4), "  σ:", round(math.sqrt(var), 4),
      "  rv_discrete:", round(deliveries.mean(), 4), round(deliveries.var(), 4))
assert close([mu, var, math.sqrt(var)], answers("P2.3"))

def portfolio(w, ex, ey, sx, sy, sxy):
    """Expected return and σ of w in X and 1 − w in Y."""
    return w * ex + (1 - w) * ey, math.sqrt(w ** 2 * sx ** 2 + (1 - w) ** 2 * sy ** 2 + 2 * w * (1 - w) * sxy)
print("portfolio (P2.4):", [round(v, 4) for v in portfolio(0.5, 8, 3, 12, 4, -10)])
assert close(portfolio(0.5, 8, 3, 12, 4, -10), answers("P2.4"))

print("P(at least 2 of 20), p = 0.1:", round(binom.sf(1, 20, 0.1), 4))
print("mean and σ of B(400, 0.1):", binom.mean(400, 0.1), binom.std(400, 0.1))
print("Poisson, 6 an hour → μ = 1 in 10 minutes; P(X ≤ 1):", round(poisson.cdf(1, 6 * 10 / 60), 4))
print("hypergeometric P(0 faulty), 3 from 15 with 4 faulty:", round(hypergeom.pmf(0, M=15, n=4, N=3), 4))
print("normal P(45 ≤ X ≤ 60):", round(norm.cdf(60, 50, 5) - norm.cdf(45, 50, 5), 4))
print("demand exceeded 10% of weeks:", round(norm.ppf(0.9, 200, 20), 2))
assert close([binom.sf(1, 20, 0.1)], answers("P2.5")) and close([binom.mean(400, 0.1), binom.std(400, 0.1)], answers("P2.6"))
assert close([poisson.cdf(1, 1)], answers("P2.7")) and close([hypergeom.pmf(0, 15, 4, 3)], answers("P2.8"))
assert close([norm.cdf(60, 50, 5) - norm.cdf(45, 50, 5)], answers("P2.9"))
assert close([norm.ppf(0.9, 200, 20)], answers("P2.10"), 1e-6)
assert close([0.45 + 0.30 - 0.12], answers("P2.1")) and close([0.3 * 0.04 / (0.7 * 0.01 + 0.3 * 0.04)], answers("P2.2"))

### Move b: the area grows

Order values, normal with mean $80 and standard deviation $16. The shaded area is P(a ≤ X ≤ b), the probability an
order falls between a and b. At a = 72 and b = 100 it's 0.5858 (the z table gives 0.5859, from rounded z values).
Drag b to 112, two standard deviations above the mean, and watch the area grow to 0.6687.

In [ ]:
def normal_area(mu=80.0, sigma=16.0, a=72.0, b=100.0):
    xs = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 401)
    lo, hi = min(a, b), max(a, b)
    area = norm.cdf(hi, mu, sigma) - norm.cdf(lo, mu, sigma)
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.15)
    ax.plot(xs, norm.pdf(xs, mu, sigma), color=CHALK)
    band = (xs >= lo) & (xs <= hi)
    ax.fill_between(xs[band], norm.pdf(xs[band], mu, sigma), color=GOLD, alpha=0.55, lw=0)
    ax.axvline(mu, color=SKY, lw=1.6, alpha=0.8)
    ax.set_xlabel(r"order value (\$)"); ax.set_yticks([]); ax.set_ylim(bottom=0)
    fig.suptitle(f"P({lo:g} ≤ X ≤ {hi:g}) = {area:.4f}   (μ = {mu:g}, σ = {sigma:g})", x=0.07, ha="left", fontsize=15)
    footnote(fig); plt.show()

assert round(norm.cdf(100, 80, 16) - norm.cdf(72, 80, 16), 4) == 0.5858
assert round(norm.cdf(112, 80, 16) - norm.cdf(72, 80, 16), 4) == 0.6687

In [ ]:
interact(normal_area, mu=FloatSlider(value=80, min=40, max=120, step=1, description="μ ($)"),
         sigma=FloatSlider(value=16, min=4, max=30, step=1, description="σ ($)"),
         a=FloatSlider(value=72, min=0, max=160, step=1, description="a ($)"),
         b=FloatSlider(value=100, min=0, max=160, step=1, description="b ($)"));

## Tier 3 · Stretch

Combine two ideas, choose the distribution, spot the trap, work backwards: a profit that is a linear function of a
random variable, the hypergeometric against the binomial, σ from a percentile, the normal approximation with and
without the continuity correction, Poisson counts against exponential gaps, a portfolio whose correlation turns
positive, a break-even probability, and a fraud alert that is right only half the time.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**A linear function** aX + b has mean a E(X) + b and standard deviation |a| σ: the b shifts, it doesn't spread.
**Working backwards** from an area is `norm.ppf`. **The normal approximation** of a binomial uses μ = np and
σ = √(np(1 − p)); the continuity correction moves the cut to the edge of the bar, x + 0.5. The picture below shows
why: the bar for 25 runs from 24.5 to 25.5.

In [ ]:
x, f = np.arange(5), np.array([0.10, 0.20, 0.30, 0.25, 0.15])                              # P3.1
m, s = (x * f).sum(), math.sqrt(((x - (x * f).sum()) ** 2 * f).sum())
print("profit: mean", round(120 * m - 100, 2), " σ", round(120 * s, 2))
assert close([120 * m - 100, 120 * s], answers("P3.1"), 1e-6)

print("no defective in 5 of 12 (3 defective): hypergeometric", round(hypergeom.pmf(0, 12, 3, 5), 4),
      "  binomial", round(binom.pmf(0, 5, 3 / 12), 4))
assert close([hypergeom.pmf(0, 12, 3, 5), binom.pmf(0, 5, 0.25)], answers("P3.2"))

z = norm.ppf(0.975)                                                                          # P3.3
print("z for the top 2.5%:", round(z, 6), "→ σ =", round((539.2 - 500) / z, 4), "g;  with the table's z of 1.96:",
      round((539.2 - 500) / 1.96, 4), "g")
assert close([(539.2 - 500) / round(z, 2)], answers("P3.3"))

print("no order in 30 minutes at 4 an hour: Poisson", round(poisson.pmf(0, 2), 4), "  exponential",
      round(expon.sf(30, scale=15), 4))
assert close([poisson.pmf(0, 2), expon.sf(30, scale=15)], answers("P3.5"))
print("break-even probability:", 200 / 2000, "   P(fraud | alert):", 0.02 * 0.98 / (0.02 * 0.98 + 0.98 * 0.02))
assert close([0.08 * 2000, 200 / 2000], answers("P3.7")) and close([0.5], answers("P3.8"))

In [ ]:
n, p = 100, 0.3                                                                              # P3.4
mu, sd = n * p, math.sqrt(n * p * (1 - p))
approx = [norm.cdf(25.5, mu, sd), norm.cdf(25, mu, sd), binom.cdf(25, n, p)]
assert close(approx, answers("P3.4"))
k = np.arange(12, 49)
fig, ax = plt.subplots(figsize=(11.5, 4.4))
fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.15)
ax.bar(k, binom.pmf(k, n, p), color=[GOLD if v <= 25 else CHALK for v in k], width=0.85)
xs = np.linspace(12, 48, 400); ax.plot(xs, norm.pdf(xs, mu, sd), color=SKY)
ax.axvline(25.5, color=GOLD, ls="--", lw=1.6); ax.axvline(25, color=CORAL, ls=":", lw=1.6)
ax.set_xlabel("customers who buy"); ax.set_ylabel("probability")
fig.suptitle(f"P(X ≤ 25): exact {approx[2]:.4f} · normal at 25.5 {approx[0]:.4f} · at 25 {approx[1]:.4f} (coral)",
             x=0.07, ha="left", fontsize=15)
footnote(fig); plt.show()

### Turn the correlation: diversification shrinks

Half in a stock fund (σ 14%) and half in a bond fund (σ² 8.76, so σ ≈ 2.96%). With the video's correlation, −0.49
(σxy = −20.4), the mix's σ is 6.40%, well below the average of the two, 8.48%. Drag ρ up: at +0.5 it's 7.85%, and at +1 the mix is
no safer than the average.

In [ ]:
RHO_FUNDS = round(-20.4 / (14 * math.sqrt(8.76)), 3)                     # −0.492: the joint table's ρ

def mix_risk(rho=RHO_FUNDS):
    sx, sy = 14, math.sqrt(8.76)
    rhos = np.linspace(-1, 1, 201)
    sig = np.sqrt(0.25 * sx ** 2 + 0.25 * sy ** 2 + 2 * 0.25 * rhos * sx * sy)
    now = math.sqrt(0.25 * sx ** 2 + 0.25 * sy ** 2 + 2 * 0.25 * rho * sx * sy)
    fig, ax = plt.subplots(figsize=(10, 4.2))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.16)
    ax.plot(rhos, sig, color=VIOLET); ax.axhline((sx + sy) / 2, color=DIM, ls="--", lw=1.4)
    ax.scatter([rho], [now], s=120, color=GOLD, zorder=3)
    ax.set_xlabel("correlation ρ"); ax.set_ylabel("σ of the 50/50 mix (%)"); ax.set_ylim(0, 10)
    fig.suptitle(f"ρ = {rho:+.2f}".replace("-", "−") + f": σ = {now:.2f}%   (average of the two σ: {(sx + sy) / 2:.2f}%, dashed)",
                 x=0.08, ha="left", fontsize=15)
    footnote(fig, "our own scenario · computed"); plt.show()

assert close([math.sqrt(0.25 * 196 + 0.25 * 8.76 + 2 * 0.25 * 0.5 * 14 * math.sqrt(8.76))], answers("P3.6"))
assert round(math.sqrt(0.25 * 196 + 0.25 * 8.76 + 2 * 0.25 * RHO_FUNDS * 14 * math.sqrt(8.76)), 2) == 6.40   # the video's 6.40

In [ ]:
interact(mix_risk, rho=FloatSlider(value=RHO_FUNDS, min=-1, max=1, step=0.001, readout_format="+.2f", description="ρ"));

## Tier 4 · The case: Halverton Hotels' overbooking decision

Halverton Hotels runs a 200-room city hotel (a fictional company; our own data). On a sold-out night it has always
held exactly 200 bookings, and about one booked guest in twenty doesn't arrive, so rooms sit empty. Each filled room
earns a margin of $160; walking a booked guest to another hotel when the rooms run out costs about $450. The
question: **how many bookings should it accept on a sold-out night?** Here are last year's 250 sold-out nights, one
row each:

| column | what it is |
|---|---|
| `date` | the night (2025) |
| `weekday` | the night's weekday; Fri and Sat are the weekend nights |
| `bookings` | bookings held that night: always 200 |
| `no_shows` | booked guests who did not arrive |
| `walk_ins` | guests without a booking who asked for a room |

The next cell builds the table from seeded random generators, so everyone gets exactly the same 250 nights (the same
as the spreadsheet's Case study tab), and checks a few of its numbers.

In [ ]:
ROOMS, NIGHTS, SHOW_P, ROOM_VALUE, WALK_COST = 200, 250, 0.05, 160, 450

def build_case():
    r = random.Random(1977)                                    # no-shows: each of 200 booked guests, p = 0.05
    no_shows = [sum(1 for _ in range(ROOMS) if r.random() < SHOW_P) for _ in range(NIGHTS)]
    w = random.Random(2025)                                    # walk-ins: Poisson, mean 2.5
    def pois(mu):
        L, k, p = math.exp(-mu), 0, 1.0
        while True:
            p *= w.random()
            if p <= L: return k
            k += 1
    walk_ins = [pois(2.5) for _ in range(NIGHTS)]
    d = random.Random(250)                                     # which 250 nights of 2025 sold out
    days = sorted(d.sample(range(365), NIGHTS))
    dates = [datetime.date(2025, 1, 1) + datetime.timedelta(days=k) for k in days]
    return pd.DataFrame({"date": [x.isoformat() for x in dates], "weekday": [x.strftime("%a") for x in dates],
                         "bookings": [ROOMS] * NIGHTS, "no_shows": no_shows, "walk_ins": walk_ins})

df = build_case()
assert len(df) == 250 and df["no_shows"].sum() == 2500 and df["no_shows"].mean() == 10
assert round(df["no_shows"].var(), 3) == 9.502 and (df["no_shows"].min(), df["no_shows"].max()) == (3, 19)
assert (df["no_shows"] <= 9).sum() == 113 and set(df["bookings"]) == {200}
print(f"{len(df)} nights, {df.shape[1]} columns: the same data as the course's case.")
df.head(8)

### How you'd do this in Python

With the nights in a DataFrame, each step of the case is a line or two: `mean()` and `var()` (pandas divides by
n − 1, like VAR.S) for the no-show distribution; `binom.pmf` for the model's expected counts; `binom.cdf` for the risk
of walking a guest; and a loop over the number of extra bookings for the expected gain. These cells reproduce the
video's case numbers. Then it's your turn.

In [ ]:
ns = df["no_shows"]
p_hat = ns.mean() / ROOMS
print("mean", ns.mean(), "  variance", round(ns.var(), 3), "  p̂ =", p_hat, "  binomial variance np(1 − p):",
      round(ROOMS * p_hat * (1 - p_hat), 3))
k = np.arange(0, 26)
fig, ax = plt.subplots(figsize=(11.5, 4.2))
fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.15)
ax.bar(k, [(ns == v).sum() for v in k], color=CHALK, width=0.8, label="nights in the data")
ax.scatter(k, NIGHTS * binom.pmf(k, ROOMS, p_hat), color=SKY, s=55, zorder=3, label="binomial(200, 0.05) × 250")
ax.set_xlabel("no-shows on a sold-out night"); ax.set_ylabel("nights"); ax.legend(frameon=False)
fig.suptitle("Halverton Hotels · 250 sold-out nights: the binomial fits", x=0.07, ha="left", fontsize=15)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

The expected gain of accepting 200 + k bookings: each extra booking fills a room only if someone doesn't show, and
walks a guest when more booked guests arrive than there are rooms. The video's Python, line for line:

In [ ]:
from scipy.stats import binom
gain = {}
for k in range(16):                                            # k extra bookings
    walked = sum((k - j) * binom.pmf(j, 200 + k, 0.05) for j in range(k))     # expected guests walked
    filled = 0.95 * (200 + k) - walked                                          # expected rooms filled
    gain[k] = 160 * (filled - 190) - 450 * walked
best = max(gain, key=gain.get); print(f"best: {best} extra bookings · ${gain[best]:.2f} a night")

In [ ]:
assert best == 8 and round(gain[8], 2) == 997.37
assert [round(gain[k]) for k in range(13)] == [0, 152, 304, 455, 602, 740, 861, 952, 997, 985, 906, 757, 542]
fig, ax = plt.subplots(figsize=(11.5, 4.2))
fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.15)
ks = list(gain)
ax.bar(ks, [gain[k] for k in ks], color=[GOLD if k == best else CHALK for k in ks], width=0.75)
ax.set_xlabel("extra bookings k"); ax.set_ylabel(r"expected gain (\$ a night)")
fig.suptitle(f"The peak: {best} extra bookings, ${gain[best]:.0f} a night more than not overbooking".replace("$", r"\$"),
             x=0.08, ha="left", fontsize=15)
footnote(fig, "a fictional company · our own data · computed"); plt.show()

### Your turn: tier 4

Reproduce the video's case numbers, then go further: a higher cost of walking a guest, weekend nights against the
rest, and the walk-ins. Use the DataFrame `df` and `binom` in a cell of your own (press **+** to add one), then type
your answers.

In [ ]:
practice("P4.1")

In [ ]:
practice("P4.2")

In [ ]:
practice("P4.3")

In [ ]:
practice("P4.4")

In [ ]:
practice("P4.5")

In [ ]:
practice("P4.6")

In [ ]:
practice("P4.7")

In [ ]:
practice("P4.8")

### The decision moves with its inputs

The gain table again, with sliders for the cost of walking a guest and the no-show probability. At $450 and
p = 0.05 the best is 8 extra bookings. Raise the walk cost to $1,000 and it falls to 7; a no-show rate of 4.5% or
5.5% moves it to 7 or 9. Near the top the curve is flat, so being one off costs little.

In [ ]:
def gain_table(walk=450, p=0.05, value=160):
    ks = np.arange(16)
    walked = np.array([sum((k - j) * binom.pmf(j, ROOMS + k, p) for j in range(k)) for k in ks])
    g = value * ((1 - p) * (ROOMS + ks) - walked - (1 - p) * ROOMS) - walk * walked
    b = int(ks[g.argmax()])
    fig, ax = plt.subplots(figsize=(11.5, 4.2))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.15)
    ax.bar(ks, g, color=[GOLD if k == b else CHALK for k in ks], width=0.75)
    ax.axhline(0, color=DIM, lw=1)
    ax.set_xlabel("extra bookings k"); ax.set_ylabel(r"expected gain (\$ a night)")
    fig.suptitle(f"walk cost \\${walk:,}, p = {p:.3f}: best {b} extra, \\${g.max():.0f} a night; "
                 f"a guest walked on {binom.cdf(b - 1, ROOMS + b, p):.0%} of nights" if b else
                 f"walk cost \\${walk:,}, p = {p:.3f}: don't overbook", x=0.08, ha="left", fontsize=15)
    footnote(fig, "a fictional company · our own data · computed"); plt.show()

In [ ]:
interact(gain_table, walk=IntSlider(value=450, min=100, max=2000, step=50, description="walk ($)"),
         p=FloatSlider(value=0.05, min=0.02, max=0.10, step=0.005, readout_format=".3f", description="p"),
         value=IntSlider(value=160, min=50, max=400, step=10, description="room ($)"));

### The tier 4 answers, from the DataFrame

Try the questions first. This cell works every tier 4 answer out from `df` with pandas and `scipy.stats` and checks
it against the practice set: open it when you've finished, or when you're stuck.

In [ ]:
def best_k(walk=WALK_COST, value=ROOM_VALUE, p=SHOW_P):
    g = {k: value * ((1 - p) * (ROOMS + k) - w - (1 - p) * ROOMS) - walk * w
         for k in range(16) for w in [sum((k - j) * binom.pmf(j, ROOMS + k, p) for j in range(k))]}
    b = max(g, key=g.get)
    return b, g[b]

weekend = df["weekday"].isin(["Fri", "Sat"])
p_ = df["no_shows"].mean() / df["bookings"].mean()
b8, g8 = best_k()
mine = {
    "P4.1": [df["no_shows"].mean(), df["no_shows"].var(), p_],
    "P4.2": [ROOMS * p_ * (1 - p_)],
    "P4.3": [binom.cdf(7, 208, p_), binom.cdf(9, 210, p_)],
    "P4.4": [b8, g8],
    "P4.5": [best_k(walk=1000)[0]],
    "P4.6": [df.loc[~weekend, "no_shows"].mean(), df.loc[weekend, "no_shows"].mean(),
             df.loc[~weekend, "no_shows"].std(), df.loc[weekend, "no_shows"].std()],
    "P4.7": [df["walk_ins"].mean(), poisson.sf(1, df["walk_ins"].mean())],
    "P4.8": [b8, round(binom.cdf(b8 - 1, ROOMS + b8, p_), 2),
             round(sum((b8 - j) * binom.pmf(j, ROOMS + b8, p_) for j in range(b8)), 2)],
}
for k, got in mine.items():
    ok = all(right(float(g), w, Q[k]["tol"]) for g, w in zip(got, answers(k)))
    assert ok and len(got) == len(answers(k)), (k, got, answers(k))
    print(f"{k}: " + " · ".join(f"{p}: {show(round(float(g), 4))}" for p, g in zip(Q[k]["parts"], got)))
print("\nEvery tier 4 answer worked out from the DataFrame matches the practice set.")

## Sources
Every exercise, data set and company here is our own (Halverton Hotels is fictional). The chapter order follows a
standard first course in business statistics (probability, discrete and continuous distributions); the definitions
agree with OpenStax, Introductory Business Statistics 2e and Introductory Statistics 2e (retrieved 2026-11-06).

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/bs02_distributions/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("bs02_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EXERCISES:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["tol"], _q["function"], _q["tier"], _q["group"]) == \
               (_e["q"], _e["working"], _e["tol"], _e["function"], _e["tier"], _e["group"]), _e["id"]
        assert close(_q["answer"], [float(v) for v in _flat(_e["answer"])]), _e["id"]
    assert len(_P.EXERCISES) == len(EX)
    assert all(df[c].tolist() == _P.CASE[c] for c in _P.CASE)
    print("matches the course's practice set")